<a href="https://colab.research.google.com/github/Bokenasubi-T/pomacea-distribution-jp/blob/main/notebooks/02_map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. Map of cleaned records by period

- Input: `data/pomacea_jp_clean.csv` (created by `01_download_and_clean.ipynb`)
- Output: `outputs/pomacea_jp_clean_map.html`

In [ ]:
# Load the cleaned data from Google Drive
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")
base = "/content/drive/MyDrive/Colab Notebooks/pomacea"
clean = pd.read_csv(f"{base}/data/pomacea_jp_clean.csv")

print(len(clean), "records")
print(clean["period"].value_counts())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
260 records
period
2019-2026    126
2011-2018     87
Unknown       31
-2010         16
Name: count, dtype: int64


In [ ]:
# Draw a map of the cleaned data
import folium
import os

colors = {"-2010":"blue", "2011-2018":"orange", "2019-2026":"red", "Unknown":"gray"}

m = folium.Map(location=[36, 138], zoom_start=5, tiles=None)
folium.TileLayer(
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr='<a href="https://maps.gsi.go.jp/development/ichiran.html">地理院タイル</a>',
    name="GSI pale map",
).add_to(m)

# One layer per period, with the number of records in its name
counts = clean["period"].value_counts()
groups = {p:folium.FeatureGroup(name=f"{p}(n={counts.get(p, 0)})").add_to(m) for p in colors}

for _, row in clean.iterrows():
  p = row["period"]
  folium.CircleMarker(
      location=[row["decimalLatitude"], row["decimalLongitude"]],
      radius=5,
      color=colors[p],
      fill=True,
      fill_color=colors[p],
      popup=f"year: {row['year']} / {row['basisOfRecord']}",
  ).add_to(groups[p])

folium.LayerControl(collapsed=False).add_to(m)

# Save as HTML
os.makedirs(f"{base}/outputs", exist_ok=True)
m.save(f"{base}/outputs/pomacea_jp_clean_map.html")
m